# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print title and description
print(metadata.name)
print(metadata.description)

## 2. Data Overview
Review available record sets, their `@id`s, and contained fields and columns (by `@id`).

In [ ]:
# List all record sets in the dataset and their fields/columns by @id
record_sets = [r for r in metadata.record_sets]
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', '<no name>')}")
    print(f"  Description: {rs.get('description', '<no description>')}")
    print("  Fields and columns:")
    for field in rs.get('fields', []):
        print(f"   • Field @id: {field['@id']}")
    for column in rs.get('columns', []):
        print(f"   • Column @id: {column['@id']}")
    print("-")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field or column `@id`s identified above.

You must set `record_sets_to_load` as the list of `@id` for the record sets to load. In this dataset, the main record set contains the protein quantification table.

In [ ]:
# Identify record sets to load by @id
# You may use the output from the previous overview cell to find the correct @id
record_sets_to_load = [rs['@id'] for rs in metadata.record_sets]
print(f"Record sets to load: {record_sets_to_load}")

dataframes = {}
for record_set_id in record_sets_to_load:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set: {record_set_id}, rows: {len(df)} columns: {df.shape[1]}")
        print(f"Sample columns: {df.columns[:10].tolist()}")
    except Exception as err:
        print(f"Could not load records for {record_set_id}: {err}")

# For typical proteomics tables, there is usually only one main record set. Use the first record set for demonstration:
main_record_set = record_sets_to_load[0] if record_sets_to_load else None

# Show top rows and columns of the first (main) record set if loaded
if main_record_set in dataframes:
    print(f"\nFirst 5 rows from record set @{main_record_set}:")
    display(dataframes[main_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps. For demonstration, we analyze protein abundance or peptide count (choose a numeric field/column by its `@id` from the data overview).

In [ ]:
# Identify a typical numeric column by @id, e.g. peptide count/abundance.
# Update this with the actual @id discovered in your dataset; for demonstration, we'll guess a common one:
import numpy as np
main_df = dataframes[main_record_set].copy()
# Try to choose a likely numeric field: we'll try common column names
candidate_numeric_fields = [col for col in main_df.columns if any(x in col.lower() for x in ['abundance', 'count', 'mw', 'peptide'])]
if candidate_numeric_fields:
    numeric_field_id = candidate_numeric_fields[0]
else:
    numeric_field_id = main_df.columns[0]
print(f"Selected numeric field for analysis: {numeric_field_id}")

# Convert to numeric, forcing errors to NaN for robustness
main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')

threshold = np.nanquantile(main_df[numeric_field_id], 0.75)
filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (upper quartile): {len(filtered_df)} records")

# Normalize field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - np.nanmean(filtered_df[numeric_field_id])) / np.nanstd(filtered_df[numeric_field_id])
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Attempt to group by another field; common categorical fields: description, accession, etc.
candidate_group_fields = [col for col in main_df.columns if any(x in col.lower() for x in ['description', 'accession', 'sample'])]
if candidate_group_fields:
    group_field_id = candidate_group_fields[0]
    print(f"Grouping by field: {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
    print(grouped_df.head())
else:
    print("No suitable group field found.")

## 5. Visualization
Visualize the distribution of the selected numeric field or the relationship between fields. For example, plot a histogram or a boxplot for the abundance/count field, optionally grouped by the chosen field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(main_df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot grouped by group_field if present
if candidate_group_fields:
    group_field_id = candidate_group_fields[0]
    top_groups = main_df[group_field_id].value_counts().nlargest(5).index.tolist()
    subset = main_df[main_df[group_field_id].isin(top_groups)]
    plt.figure(figsize=(10,5))
    sns.boxplot(data=subset, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id} (top 5 groups)")
    plt.show()

## 6. Conclusion
This notebook loaded and explored the mass spectrometry protein dataset using the Croissant schema and `mlcroissant`. You can further refine the analysis and repeat the workflow for other fields and record sets by referencing their `@id` identifiers.